## Build and save IRIS Full-Disk Mosaics

Import Statements

In [ ]:
from iris_mosaics.read_full_disk_mosaic import build_mosaic_regrid
%reload_ext autoreload
%autoreload 2

import pathlib as pl
import numpy as np
import matplotlib.pyplot as plt
import pickle
from matplotlib import colors
import astropy.units as u
from iris_mosaics import wcs_to_bins, spectral_plot, read_sg_image, read_sg_image_lvl1, build_mosaic, build_mosaic_sav
import reproject
import iris_mosaics as iris_fdm
from IPython.display import display, Math, Markdown
from astropy.visualization import quantity_support
quantity_support()

Load individual FDM images

In [ ]:
# Path of IRIS spectrograph images for a single mosaic
date_string = '20240811'
fits_path = pl.Path(fr'D:\IRIS data\deep_mosaics\{date_string}\level_15_rebinned')
fits_files = list(fits_path.glob('*.fits'))

# sav_path = pl.Path(r'D:\IRIS data\deep_mosaics\20190912\JP_new_bg_subtracted\20190912_BGTest_Level1p5')
# sav_files = list(sav_path.glob('*.sav'))

Assemble mosaic

In [ ]:
file_num = 1000
w_0, hdu_0, _ = read_sg_image(fits_files[file_num],'fuv2')
img_0 = hdu_0[0].data

In [ ]:
plot_min = -5

plt.figure(figsize=(15,5))
plt.imshow(img_0,
           vmin=plot_min,
           vmax=np.nanpercentile(img_0,99.5),
           origin='lower'
           )
plt.colorbar(pad=0.01).set_label('DN')
plt.xticks([])
plt.yticks([])
start_index = 740
end_index = 1018
plt.axvline(start_index, color='r')
plt.axvline(end_index, color='r')

In [ ]:
# Create masks for the areas of interest (Si IV 1394 & 1403).
# sl_sg_img = slice(5,~5), slice(720,None)
sl_sg_img = slice(None), slice(start_index, end_index)

# Si IV 1394 and 1403 image dimension lengths
num_y = img_0[sl_sg_img].shape[0]
num_x = img_0[sl_sg_img].shape[1]

In [ ]:
# Length of number of images dimension
num_imgs = len(fits_files)

# Read in spectrograph images, selecting just part of the FUV2 region
sg_imgs = np.empty((num_imgs, num_y, num_x))

for i, file in enumerate(fits_files):
    # Read in full image
    w, hdu, _ = read_sg_image(file,'fuv2')
    sg_img = hdu[0].data
    # Crop to only section of image containing the line of interest
    sg_imgs[i] = sg_img[sl_sg_img]

In [ ]:
sg_mean_image = np.nanmean(sg_imgs, axis=0)

In [ ]:
plot_min = -5

plt.figure(figsize=(10,10))
plt.imshow(sg_mean_image,
           vmin=plot_min,
           vmax=np.nanpercentile(sg_mean_image,99.9),
           origin='lower'
           )
plt.colorbar(pad=0.01).set_label('DN')
# plt.xticks([])
plt.yticks([])

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(np.nanmean(sg_mean_image, axis=0))
plt.axhline(y=0, color='r', linewidth=1, linestyle='dotted')

In [ ]:
%%time
# Build mosaic from spectrograph images
global_data, global_wcs = build_mosaic(fits_files)

# Build mosaic from .sav files from LMSAL, but using WCS info from our fits file version of the mosaic (.sav WCS isn't right...)
# global_data, global_wcs = build_mosaic_sav(fits_files, sav_files)

# For cropping out the shadowed data, use slices
# global_data, global_wcs = build_mosaic(files, (slice(None, 400), slice(None)))
# global_data, global_wcs = build_mosaic(files)

#### Mosaic plots

In [ ]:
spatial_wcs = global_wcs.deepcopy().dropaxis(0).swapaxes(0, 1)

In [ ]:
w_full = np.arange(0, global_data.shape[-1])
w_full, _, _ = global_wcs.all_pix2world(w_full, 0, 0, 0)
w_full = (w_full * u.m).to(u.angstrom)

In [ ]:
global_data_spectrum = np.nanmean(global_data, axis=(0,1))

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(w_full, global_data_spectrum)
plt.axhline(y=0, color='r', linewidth=1, linestyle='dotted')
max_index = np.nanargmax(global_data_spectrum)
plt.axvline(w_full[max_index], color='r', linewidth=1, linestyle='dotted')
w_full[max_index]   # Not wavelength calibrated

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(global_data_spectrum[4:])
plt.plot(np.nanmean(sg_mean_image, axis=0))
plt.axhline(y=0, color='r', linewidth=1, linestyle='dotted')

In [ ]:
i = max_index
plt.figure(figsize=(15,10))
ax = plt.subplot(projection=spatial_wcs)
img = ax.imshow(np.nan_to_num(global_data)[...,i].T,
                # vmin=-1,
                # vmax=10,
                norm=colors.PowerNorm(.5, vmin=0, vmax=400),
                aspect=(global_data.shape[0] / global_data.shape[1]),
                )
for j in (0, 1):
    ax.coords[j].set_format_unit(u.arcsec)   # display in arcsec
cbar = plt.colorbar(img)
cbar.set_label('DN')
ax.set_title(f'Full Disk Mosaic at {w_full[i]:.2f}');

In [ ]:
# Define dictionary

fdm_dict = dict(data=global_data, wcs=global_wcs)

Save FDM and WCS

In [ ]:
# Save FDM dictionary

save_path = pl.Path(fr'D:\IRIS data\deep_mosaics\{date_string}')
with open(save_path / 'level_15_fdm.pickle', 'wb') as fh:
    pickle.dump(fdm_dict, fh)
